# Week 07 · 天気ハッカソン / Weather Hackathon

チーム全員の氏名 / Team members：
自分の担当 / My contribution：

天気予報を使って、誰かの「いつ・どうする？」に視覚的に答えるNotebookを作ろう。2〜3人で制作し、最後に全チームが実演する。

Build a weather-assistant notebook with a visual answer to someone’s question. Work in teams of 2–3 and give a short demo at the end.

## 1. 予報を読み込む / Load the forecast

最初の2つのコードセルは準備用。そのまま実行。追加インストール・APIキーは不要。接続できなければ、同じフォルダのweather-response.jsonを使用。取得成功時はこのファイルを更新します。

Run the first two code cells as provided. No extra packages or API key. Keep weather-response.json beside this notebook as a fallback. A successful request updates that file.

Source: https://open-meteo.com/en/docs
Time: Asia/Tokyo; temp: °C; rain: precipitation probability (%).

In [ ]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode

url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
    "latitude": 35.86, "longitude": 139.97,
    "hourly": "temperature_2m,precipitation_probability",
    "forecast_hours": 24, "timezone": "Asia/Tokyo"
})
from datetime import datetime, timezone

try:
    with urlopen(url, timeout=10) as response:
        data = json.load(response)
    data["_source"] = {"url": url,
                       "retrieved_at": datetime.now(timezone.utc).isoformat()}
    with open("weather-response.json", "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
    print("Live API / APIから取得")
except (OSError, ValueError):
    with open("weather-response.json", encoding="utf-8") as file:
        data = json.load(file)
    print("Saved JSON / 保存した予報を使用")

print("Retrieved:", data.get("_source", {}).get("retrieved_at", "see forecast dates"))

## 2. 辞書のリストへ / A list of dictionaries

同じ番号iで時刻・気温・降水確率を組にしてforecastを作る。None（値なし）の時間は除きます。この準備セルもそのまま実行。

Pair time, temperature and rain probability at index i. Skip missing values. Run this setup cell as provided.

In [ ]:
forecast = []
hourly = data["hourly"]
for i in range(len(hourly["time"])):
    temp = hourly["temperature_2m"][i]
    rain = hourly["precipitation_probability"][i]
    if temp is not None and rain is not None:
        forecast.append({"time": hourly["time"][i],
                         "temp": temp, "rain": rain})
print(len(forecast), "hours / 時間分")
print("Forecast:", forecast[0]["time"], "to", forecast[-1]["time"])

## 3. 条件を、自分たちの目的へ / Make the rule your own

使う人 / Our user：
助ける判断 / Our question：

まず実行。次に、max_rain（降水確率の上限%）やmin_temp（最低気温°C）、ifの条件を変えよう。

Run it, then adapt the values and condition to your user. Write a Markdown explanation of the rule.

In [ ]:
max_rain = 30
min_temp = 10
chosen = []
for hour in forecast:
    if hour["rain"] <= max_rain and hour["temp"] >= min_temp:
        chosen.append(hour)
print(len(chosen), "hours match")

## 4. 結果を見せる / Show your result

次の表示コードは配布用。そのまま実行してよい。関数の中身は今日の課題ではありません。
The display helper is provided. Run it unchanged. Defining the helper is not part of today’s task.

In [ ]:
from html import escape

def weather_board(forecast, chosen, min_temp, max_rain):
    board = f'<div style="font:15px/1.5 system-ui;color:#203b2d"><h3 style="margin:0">{len(chosen)} / {len(forecast)} hours match · 条件に合う時間</h3>'
    board += f'<p style="margin:4px 0 12px">Temperature ≥ {min_temp}°C · Precipitation ≤ {max_rain}% · Japan time · Matches first / 合う時間を先に表示</p>'
    board += '<div style="max-height:340px;overflow:auto"><table style="border-collapse:collapse;width:100%;text-align:left"><thead><tr style="background:#315876;color:white"><th style="padding:8px">Time / 時刻</th><th>°C</th><th>Rain %</th><th>Your rule / 条件</th></tr></thead><tbody>'
    for hour in chosen + [item for item in forecast if item not in chosen]:
        matches = hour in chosen
        color = '#dceee2' if matches else '#f1f3f2'
        label = '✓ Match / 合う' if matches else '— / 合わない'
        time = escape(hour["time"].replace("T", " "))
        board += f'<tr style="background:{color};border-bottom:1px solid white"><td style="padding:7px 8px">{time}</td><td>{hour["temp"]}</td><td>{hour["rain"]}%</td><td>{label}</td></tr>'
    board += '</tbody></table></div>'
    if not chosen:
        board += '<p>No hours match. Try another setting. / 条件を変えて試そう。</p>'
    return board + '</div>'


In [ ]:
from IPython.display import HTML, display
display(HTML(weather_board(forecast, chosen, min_temp, max_rain)))

緑の行は、あなたのfor・ifが選んだ時間。条件を変え、条件セルと表示セルをもう一度実行しよう。表の代わりにグラフや地図を作ってもよい。

Green rows show the hours selected by your code. Change a rule and rerun both the filter and display. You can use a chart or map instead.

## 5. 繰り返し使う / Let the user try again

showで表示、warmerで最低気温を1°C上げて再表示、quitで終了。入力待ちの間は実行中です。設定を変えて結果が変わるか確かめ、使う人に合う操作へ変えよう。

show displays results, warmer raises the minimum temperature by 1°C, quit ends the loop. Adapt the commands for your user.

In [ ]:
command = ""
while command != "quit":
    command = input("show / warmer / quit: ").strip().lower()
    if command == "warmer":
        min_temp += 1
    if command in ["show", "warmer"]:
        print("Minimum temperature:", min_temp, "°C")
        matches = 0
        chosen = []
        for hour in forecast:
            if hour["rain"] <= max_rain and hour["temp"] >= min_temp:
                print(hour["time"], hour["temp"], "°C", hour["rain"], "%")
                matches += 1
                chosen.append(hour)
        display(HTML(weather_board(forecast, chosen, min_temp, max_rain)))
        if matches == 0:
            print("No matching hours / 条件に合う時間はありません")

## 6. 確認と説明 / Check and explain

設定1 / Setting 1：
予想と実際の結果 / Prediction and result：

設定2 / Setting 2：
予想と実際の結果 / Prediction and result：

条件に合う時間がない場合も試す（例：min_temp = 50）。quitで終了できるか確認。
Try a no-match case too. Check that quit finishes.

結果に基づく提案 / Our recommendation：

## 出典と日付 / Source and dates
Open-Meteo: https://open-meteo.com/
API取得か保存JSONか、取得日時・予報の日付を記す。保存データを今の予報とは呼ばない。
State whether you used live API or saved data, its retrieval time and forecast dates.

## グループの進め方 / Good group work

- 最初に、使う人と目的を全員で決める。
  Agree on the user and the question before coding.
- コードを書く・結果を確かめる・表示と説明を整える。得意なことを分担し、困ったら一緒に考える。
  Share the coding, checking and visual explanation. Help each other when someone gets stuck.
- 全員がコードの変更に参加し、その変更を説明できるようにする。
  Everyone contributes a code change and understands what it does.
- まず小さく動かす。変更を一つずつ確かめ、発表前に全員で実行する。
  Get a small version working. Check one change at a time and run the notebook together before presenting.

AIはエラーの説明や小さなヒントに使ってよい。作品全体を任せず、使ったコードは全員で読み、結果を自分たちで確かめる。

AI can explain an error or give a small hint. Keep the build in your hands: read the code together and check its results yourselves.

## 発表 / Demo

60〜90秒。使う人と目的、視覚表示、条件を変えた結果を見せ、for・if・whileの一行を説明。全員が一言話す。別の発表スライドは不要。

Show the visual, change a setting and explain a line of code. Everyone speaks briefly. Present from the notebook.

## AIを使った場合 / If you used AI
何を相談し、どの結果を自分で確かめた？使わなかった場合は「使っていない」。
Briefly note the help you used and what you checked yourself.

## 宿題 / Homework
発表でもらった意見を使い、同じ道具を一つ改善する。各自のNotebookにチーム全員の氏名、自分の担当、改善点と設定2通りの確認を書く。実行済み.ipynbとweather-response.jsonをUNIPAへ提出。AIを使った場合は、用途と自分で確認したことを短く記す。

Improve the same tool using demo feedback. Each student submits an executed notebook and weather-response.json to UNIPA. Name the team, your contribution, one improvement and two tested settings. Briefly note any AI help and what you checked yourself.

締切 / Due：2026-11-18 23:59 JST

途中でも、自分で書いたコードと試した変更を残して提出する。
If unfinished, submit the code you wrote and explain what you tried.